# Tightloop

Tightloop was a way of pre-calculating all information required to do a given contraction. No tensorkit overhead, no digging and calculating, it'd all be specialized given the spaces. The usecase were "tight loops", evaluating the same contraction over and over again, like in an eigenvalue problem. It used to give a noticeable speedup but through voodoo and witchcraft, native TensorKit is now comfortably beating TightLoop on everything;

I'm keeping the notebook around for posterity.

In [1]:
using Revise, TensorKit, TensorOperations, BenchmarkTools, Strided, TupleTools, MPSKitExperimental

In [2]:
import LinearAlgebra
using Base.Threads
LinearAlgebra.BLAS.set_num_threads(1)
Strided.set_num_threads(1)
nthreads()

4

## Practical benchmarks

In [3]:
@tightloop_tensor ac_eff y[-1 -2;-3] := le[-1 2;1]*O[2 -2;3 4]*x[1 3;5]*re[5 4;-3]
@tightloop_tensor ac_eff_malloc allocator=malloc y[-1 -2;-3] := le[-1 2;1]*O[2 -2;3 4]*x[1 3;5]*re[5 4;-3]

In [4]:
virtspace = Rep[SU₂](i => 20 for i in 0:10);
ospace = Rep[SU₂](0 => 5,1 => 2);
pspace = Rep[SU₂](1 => 1);

t_le = rand(ComplexF64,virtspace*ospace',virtspace);
t_re = rand(ComplexF64,virtspace*ospace,virtspace);
t_ac = rand(ComplexF64,virtspace*pspace,virtspace);
t_o = rand(ComplexF64,ospace*pspace,pspace*ospace);

factory = ac_eff(le = (typeof(t_le),space(t_le)),re = (typeof(t_re),space(t_re)),O = (typeof(t_o),space(t_o)),x = (typeof(t_ac),space(t_ac)));
factory_malloc = ac_eff_malloc(le = (typeof(t_le),space(t_le)),re = (typeof(t_re),space(t_re)),O = (typeof(t_o),space(t_o)),x = (typeof(t_ac),space(t_ac)));

In [5]:
@benchmark $factory_malloc(le = $t_le, re=$t_re, x = $t_ac, O = $t_o)

BenchmarkTools.Trial: 1556 samples with 1 evaluation per sample.
 Range (min … max):  2.183 ms … 12.806 ms  ┊ GC (min … max): 0.00% … 46.63%
 Time  (median):     2.697 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   3.211 ms ±  1.249 ms  ┊ GC (mean ± σ):  2.51% ±  6.73%

   ▄█▆▃                                                       
  ▅████▇▄▃▄▄▅▄▄▃▃▃▃▃▂▂▃▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▂▂▁▁▁▁▂▁▁▁▁▁▂ ▃
  2.18 ms        Histogram: frequency by time        9.25 ms <

 Memory estimate: 472.77 KiB, allocs estimate: 4713.

In [6]:
@benchmark $factory(le = $t_le, re=$t_re, x = $t_ac, O = $t_o)

BenchmarkTools.Trial: 834 samples with 1 evaluation per sample.
 Range (min … max):  2.942 ms … 144.687 ms  ┊ GC (min … max):  0.00% … 96.44%
 Time  (median):     3.433 ms               ┊ GC (median):     0.00%
 Time  (mean ± σ):   6.046 ms ±  13.203 ms  ┊ GC (mean ± σ):  35.75% ± 19.92%

  █▂▃▄▃                                                        
  ████████▄▆▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄▄▁▄▄▄▄▄▁▁▄ ▇
  2.94 ms      Histogram: log(frequency) by time        57 ms <

 Memory estimate: 11.78 MiB, allocs estimate: 4717.

In [7]:
function slowcontract(;le=nothing,re=nothing,x=nothing,O=nothing)
    @tensor allocator=malloc y[-1 -2;-3] := le[-1 2;1]*O[2 -2;3 4]*x[1 3;5]*re[5 4;-3]
end
@benchmark slowcontract(le = $t_le, re=$t_re, x = $t_ac, O = $t_o)

BenchmarkTools.Trial: 1000 samples with 1 evaluation per sample.
 Range (min … max):  2.413 ms … 14.436 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     4.734 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.996 ms ±  1.186 ms  ┊ GC (mean ± σ):  1.30% ± 5.59%

  ▁▁          ▃▂▅██▇▂▁▁▂▂▂▂                                   
  ██▅▅▁▁▁▁▁▁▁▇████████████████▇▇▆█▄▆▄▅▄▇▆▅▅▄▄▅▁▁▁▁▁▁▄▄▅▁▁▅▄▆ █
  2.41 ms      Histogram: log(frequency) by time     10.6 ms <

 Memory estimate: 472.21 KiB, allocs estimate: 4716.

In [8]:
norm(slowcontract(le = t_le, re=t_re, x = t_ac, O = t_o)- factory(le = t_le, re=t_re, x = t_ac, O = t_o))

0.0

In [9]:
virtspace = Rep[SU₂×U₁×ℤ₂]((i,j,b) => 10 for i in 0:5,j in -3:3,b in 0:1);
ospace = Rep[SU₂×U₁×ℤ₂]((i,j,b) => 2 for i in 0:1,j in -1:1,b in 0:1);
pspace = Rep[SU₂×U₁×ℤ₂]((i,j,b) => 1 for i in 1:1,j in 0:0,b in 0:0);

t_le = rand(ComplexF64,virtspace*ospace',virtspace);
t_re = rand(ComplexF64,virtspace*ospace,virtspace);
t_ac = rand(ComplexF64,virtspace*pspace,virtspace);
t_o = rand(ComplexF64,ospace*pspace,pspace*ospace);

factory = ac_eff_malloc(le = (typeof(t_le),space(t_le)),re = (typeof(t_re),space(t_re)),O = (typeof(t_o),space(t_o)),x = (typeof(t_ac),space(t_ac)));

In [10]:
@benchmark $factory(le = $t_le, re=$t_re, x = $t_ac, O = $t_o)

BenchmarkTools.Trial: 101 samples with 1 evaluation per sample.
 Range (min … max):  42.642 ms … 68.625 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     47.429 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   49.513 ms ±  6.386 ms  ┊ GC (mean ± σ):  2.94% ± 5.70%

   ▇█▇                      ▂                                  
  ▅███▆▅▇▇▁▃▆▅▃▇▃▁▃▅▁▁▃▆▃▆▁▆█▁█▅▅▁▁▃▃▃▃▃▁▁▁▃▁▁▃▁▁▃▃▃▁▃▁▃▁▁▁▁▃ ▃
  42.6 ms         Histogram: frequency by time        67.2 ms <

 Memory estimate: 9.70 MiB, allocs estimate: 157226.

In [11]:
@benchmark slowcontract(le = $t_le, re=$t_re, x = $t_ac, O = $t_o)

BenchmarkTools.Trial: 108 samples with 1 evaluation per sample.
 Range (min … max):  39.951 ms … 65.738 ms  ┊ GC (min … max): 0.00% … 12.09%
 Time  (median):     44.268 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   46.658 ms ±  6.034 ms  ┊ GC (mean ± σ):  2.86% ±  5.54%

  ▃█▃▅  ▂ ▅              ▂                                     
  ███████▄██▇▁▄▅▇▇▄▁▁▅▄▅▇█▄▄▄▁▇█▅█▁▁▄▄▁▁▇▁▁▅▁▁▄▄▁▄▄▁▁▁▁▁▁▁▁▁▄ ▄
  40 ms           Histogram: frequency by time        64.6 ms <

 Memory estimate: 9.70 MiB, allocs estimate: 157229.

In [12]:
norm(slowcontract(le = t_le, re=t_re, x = t_ac, O = t_o)- factory(le = t_le, re=t_re, x = t_ac, O = t_o))

0.0

## Multithreaded benchmark

gc becomes a severe bottleneck with many julia threads

In [13]:
virtspace = Rep[SU₂](0=>50, 1=> 30, 2=> 20, 3=>10, 4=> 5);
ospace = Rep[SU₂](0=>2, 1 => 2);
pspace = Rep[SU₂](1 => 1);


In [14]:
numblocks = 100;
const leftenvs = [rand(ComplexF64,virtspace*ospace',virtspace) for i in 1:numblocks];
const rightenvs = [rand(ComplexF64,virtspace*ospace,virtspace) for i in 1:numblocks];
const mpos = [rand(ComplexF64,ospace*pspace,pspace*ospace) for i in 1:numblocks];

ac = rand(ComplexF64,virtspace*pspace,virtspace);

In [15]:
using FLoops

t_le = leftenvs[1]
t_re = rightenvs[1]
t_o = mpos[1]

const fast_factory = ac_eff_malloc(le = (typeof(t_le),space(t_le)),re = (typeof(t_re),space(t_re)),O = (typeof(t_o),space(t_o)),x = (typeof(ac),space(ac)))

function fastfun(x)
    @floop for (l,m,r) in zip(leftenvs,mpos,rightenvs)
        y = fast_factory(le = l,re = r, O = m, x = x)
        @reduce s += y
    end
    s
end

function slowfun(x)
    @floop for (le,O,re) in zip(leftenvs,mpos,rightenvs)
        @tensor allocator=malloc y[-1 -2;-3] := le[-1 2;1]*O[2 -2;3 4]*x[1 3;5]*re[5 4;-3]
        @reduce s += y
    end
    s
end

slowfun (generic function with 1 method)

In [16]:
@benchmark fastfun(ac)

BenchmarkTools.Trial: 103 samples with 1 evaluation per sample.
 Range (min … max):  31.186 ms … 208.451 ms  ┊ GC (min … max):  0.00% … 79.16%
 Time  (median):     45.494 ms               ┊ GC (median):     5.77%
 Time  (mean ± σ):   48.836 ms ±  23.589 ms  ┊ GC (mean ± σ):  12.57% ± 12.56%

     ▄▆█▃                                                       
  ▃▆▆████▄▃▁▃▁▁▁▁▁▃▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃ ▃
  31.2 ms         Histogram: frequency by time          203 ms <

 Memory estimate: 29.68 MiB, allocs estimate: 187473.

In [17]:
@benchmark slowfun(ac)

BenchmarkTools.Trial: 108 samples with 1 evaluation per sample.
 Range (min … max):  34.534 ms … 222.386 ms  ┊ GC (min … max):  0.00% … 78.46%
 Time  (median):     44.319 ms               ┊ GC (median):     6.82%
 Time  (mean ± σ):   46.481 ms ±  18.368 ms  ┊ GC (mean ± σ):  11.22% ±  9.95%

     ▂ ▃▃▇▆█▂█                                                  
  ▄▃▄█▄███████▇▃█▅▃▄▃▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃ ▃
  34.5 ms         Histogram: frequency by time         98.9 ms <

 Memory estimate: 29.63 MiB, allocs estimate: 187773.